# Pha S — Buoc 0 (BAT BUOC TRUOC TIEN): Train lai T_phi bat bien thang do

## ⛔ Khong duoc bo qua notebook nay

`T_phi` hien tai (`phi_amortized.pt`) duoc train tren du lieu tong hop co `mean~0, std~0.8-1.0`. Du lieu that co thang do hoan toan khac (RR interval ~0.8+-0.1 giay; EEG band power 1-500 uV^2).

**Da do truc tiep** tren checkpoint hien co, 300 cua so test:

| Dau vao | bias | variance | **corr voi ket qua dung** |
|---|---|---|---|
| Nguyen goc | -0.0173 | 0.00085 | 1.00 |
| Z-score | -0.0751 | 0.00334 | 0.87 |
| Doi thang do `x0.1+0.8` | 0.0055 | 0.00010 | **0.07** |

Dong cuoi (mo phong dung thang do RR interval) co bias/variance **dep nhat bang** nhung tuong quan voi dap an chi **0.07** - output da sup ve gan hang so. Neu chay Pha S ma khong sua, bang ket qua se trong hoan toan binh thuong va **khong ai phat hien ra la rac**.

**Co so ly thuyet:** TE bat bien duoi phep bien doi affine tung bien - `TE(aX+b -> cY+d) = TE(X->Y)`. Da kiem chung: KSG cho 0.1697 tren du lieu goc va 0.1697 sau khi doi thang do (giong het). Nghia la chuan hoa **khong lam mat gi ve ly thuyet**.

**Thoi gian:** ~2 gio (theo do thuc te o ablation Pha R).

Xem `docs/PHASE_S_GUIDE.md` muc 0.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import yaml, time, numpy as np, torch
from pathlib import Path

from pqrst.data.synthetic.corpus import load_corpus
from pqrst.estimators.mine.amortized import (train_amortized, AmortizedTrainConfig,
                                             AmortizedTEEstimator)
from pqrst.utils.standardize import standardize_window

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
corpus_dir = BASE / 'data' / 'interim' / 'synthetic_corpus'
train_windows = load_corpus(str(corpus_dir / 'train.npz'))
val_windows = load_corpus(str(corpus_dir / 'val.npz'))
test_windows = load_corpus(str(corpus_dir / 'test.npz'))
print(f'train={len(train_windows)}, val={len(val_windows)}, test={len(test_windows)}')

train=22950, val=4050, test=18000


## 1. Uoc tinh thoi gian (chay truoc)

In [2]:
config = AmortizedTrainConfig()
config.standardize = True
config.max_epochs = 2
start = time.time()
est_temp, hist = train_amortized(train_windows[:320], val_windows[:320], config)
print(f'Time for 2 epochs on small subset: {time.time()-start:.2f}s')


Time for 2 epochs on small subset: 15.19s


## 2. Train lai voi chuan hoa bat

In [3]:
config = AmortizedTrainConfig()
config.standardize = True
est, hist = train_amortized(train_windows, val_windows, config)
checkpoint_path = BASE / 'results' / 'checkpoints' / 'phi_amortized_standardized.pt'
est.save(str(checkpoint_path))
print('Saved checkpoint successfully.')


Saved checkpoint successfully.


## 3. ✅ XAC MINH 1 — hieu nang tren synthetic khong te di o N>=50

So bias/variance cua ban chuan hoa vs ban goc tren tap test, rieng cho N>=50 (vung `T_phi` manh theo Pha R). Neu te di dang ke -> co van de, dung chay tiep.

In [4]:
from pqrst.evaluation.grid import evaluate_estimators_on_grid
import pandas as pd
est_old = AmortizedTEEstimator.load(str(BASE / 'results' / 'checkpoints' / 'phi_amortized.pt'))
test_n50 = [w for w in test_windows if w.n_samples >= 50]
res_old = evaluate_estimators_on_grid({'Amortized': est_old}, test_n50)
res_new = evaluate_estimators_on_grid({'Amortized_Std': est}, test_n50)
old_mse = res_old['error'].apply(lambda x: x**2).mean()
new_mse = res_new['error'].apply(lambda x: x**2).mean()
print(f'Old MSE (N>=50): {old_mse:.5f}')
print(f'New MSE (N>=50): {new_mse:.5f}')


Evaluating windows: 100%|██████████| 9000/9000 [02:41<00:00, 55.73it/s]


Old MSE (N>=50): 0.00509
New MSE (N>=50): 0.00446


## 4. ✅ XAC MINH 2 — BAT BIEN THANG DO (day la muc dich chinh)

Dua cung du lieu nhung doi thang do (`x*0.1+0.8`, mo phong RR interval). Ban chuan hoa phai cho ket qua **gan nhu khong doi**: `corr > 0.99`.

Doi chieu: ban goc chi dat `corr = 0.07` o phep thu nay.

In [5]:
te_orig = []
te_scaled = []
for w in test_n50[:300]:
    x = np.zeros(w.n_samples + 1)
    y = np.zeros(w.n_samples + 1)
    x[:-1] = w.x_lag
    y[:-1] = w.y_lag
    y[1:] = w.y_t
    te_orig.append(est.estimate(x, y))
    te_scaled.append(est.estimate(x * 0.1 + 0.8, y * 0.1 + 0.8))
corr = np.corrcoef(te_orig, te_scaled)[0, 1]
print(f'Correlation after standardization: {corr:.5f}')
assert corr > 0.99


Correlation after standardization: 1.00000


## 5. Ket luan

**Chi khi CA HAI xac minh o muc 3 va 4 deu dat moi duoc sang notebook 01.** Neu khong dat: dung lai, bao cao, khong chay tiep len du lieu that.